# Freyra — Influencer Edition (T4 Colab)
Photorealistic female influencer images on a free T4 GPU.  
Run **Cell 2** (setup), then **Cell 3** (launch). That's it.


In [ ]:
# ── Cell 2 · Setup ───────────────────────────────────────────────────────
%cd /content
!git clone https://github.com/Ashutosh94g/Freyra.git 2>/dev/null || git -C Freyra pull --ff-only
%cd /content/Freyra

# Remove cupy if present — it forces numpy 2.x which breaks the pipeline
!pip uninstall -yq cupy-cuda12x cupy-cuda11x cupy 2>/dev/null; true

# PyTorch 2.4.1 + CUDA 12.1 (verified on T4)
%pip install -q torch==2.4.1 torchvision==0.19.1 --extra-index-url https://download.pytorch.org/whl/cu121

# All project requirements (gradio 3.50.2, transformers>=4.45, numpy<2.0, etc.)
%pip install -q -r requirements_versions.txt

print("\n✅ Setup complete — run Cell 3 to launch.")


In [ ]:
# ── Cell 3 · Verify GPU & VRAM ───────────────────────────────────────────
import torch

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    total_gb = props.total_memory / 1024**3
    free_gb  = (props.total_memory - torch.cuda.memory_reserved(0)) / 1024**3
    print(f'GPU  : {props.name}')
    print(f'VRAM : {total_gb:.1f} GB total  |  {free_gb:.1f} GB free')
    if total_gb < 12:
        print('⚠️  Less than 12 GB VRAM — lower resolution to 832×1152 in the UI.')
    else:
        print('✅ VRAM looks good for influencer preset.')
else:
    print('❌ No CUDA GPU — go to Runtime → Change runtime type → T4 GPU.')


In [ ]:
# ── Cell 4 · Launch ───────────────────────────────────────────────────────
%cd /content/Freyra
!python launch.py \
  --share \
  --preset influencer \
  --always-gpu \
  --unet-in-fp8-e4m3fn \
  --attention-pytorch \
  --disable-image-log


## Tips & Troubleshooting

| Problem | Fix |
|---|---|
| **OOM / CUDA out of memory** | Lower resolution to `832×1152` in the UI, or disable one LoRA |
| **`ModuleNotFoundError: numpy`** | Re-run Cell 2 |
| **Gradio URL not showing** | Wait 60–90 s — checkpoint download is ~6.6 GB |
| **Black / broken images** | Reduce Film Photography LoRA weight to 0.15 in the UI |

### T4 resolution guide
| Aspect | Resolution | Notes |
|---|---|---|
| Portrait 3:4 | `896×1152` | Default |
| Square 1:1 | `1024×1024` | Fine |
| Landscape 4:3 | `1152×896` | Fine |
| OOM fallback | `832×1152` | Also reduce LoRAs |
